# 01 — Collecte

**Cartographier les manques des ERP** · Ameth Yade · M2 SIGLIS, Université de Pau et des Pays de l'Adour

---

Un module tiers n'existe que pour une raison : quelqu'un a payé un développeur parce
que le progiciel standard ne répondait pas à son besoin. Chaque module publié est donc
la trace d'un manque, et leur distribution dessine une carte.

Ce notebook construit le jeu de données. Il suit quatre étapes :

1. vérifier ce que le site autorise, avant d'écrire la moindre requête ;
2. reconstituer la liste des catégories — c'est là que se joue la validité du corpus ;
3. relever les effectifs par catégorie × version ;
4. collecter les fiches de chaque module.

## 1. Avant tout : ce que le site autorise

Lire le `robots.txt` n'est pas une formalité. Il détermine ce qu'on a le droit de
parcourir, et donc la stratégie de collecte tout entière.

In [ ]:
import re
import requests

CONTACT = "ameth.yade@uadb.edu.sn"
USER_AGENT = f"erp-gaps-analysis/1.0 (academic research, M2 SIGLIS; contact: {CONTACT})"
BASE = "https://apps.odoo.com"

robots = requests.get(f"{BASE}/robots.txt", headers={"User-Agent": USER_AGENT}, timeout=30).text

# On n'affiche que les regles susceptibles de toucher le parcours des modules.
for line in robots.splitlines():
    if re.search(r"page|order|search|crawl-delay|user-agent|sitemap", line, re.I):
        print(line)

### Ce que ces règles impliquent

Trois interdictions concernent notre parcours :

| Règle | Portée |
|---|---|
| `Disallow: */page/*/*` | un segment `/page/` **suivi d'un autre segment** |
| `Disallow: *?*page=` | la pagination passée en **paramètre d'URL** |
| `Disallow: *?*order=` | tout **tri** par paramètre |

La pagination réelle du store est `/apps/modules/browse/page/7` : le numéro est un
segment **terminal**, sans segment après lui. Elle ne tombe donc pas sous
`*/page/*/*`. Aucun paramètre `page=` ni `order=` n'est employé ici.

Le paramètre `?series=`, lui, n'est pas restreint.

**Aucun `Crawl-delay` n'est déclaré** — ce qui ne dispense de rien. Le collecteur
s'impose 1,5 seconde entre deux requêtes.

In [ ]:
def est_autorisee(url_path: str) -> bool:
    """Verifie une URL contre les trois regles qui nous concernent."""
    if re.search(r"/page/[^/]+/", url_path):   # */page/*/*
        return False
    if re.search(r"\?.*\bpage=", url_path):    # *?*page=
        return False
    if re.search(r"\?.*\border=", url_path):   # *?*order=
        return False
    return True


for chemin in [
    "/apps/modules/browse",
    "/apps/modules/browse/page/7",
    "/apps/modules/category/Accounting/browse/page/165?series=17.0",
    "/apps/modules/browse?page=7",          # interdit
    "/apps/modules/browse?order=Downloads", # interdit
    "/apps/modules/browse/page/7/extra",    # interdit
]:
    print(f"  {'OK ' if est_autorisee(chemin) else 'NON'}  {chemin}")

## 2. Le client HTTP

Trois garanties, dans l'ordre d'importance.

**Le cache disque d'abord.** Une URL n'est jamais téléchargée deux fois. C'est ce qui
permet de retravailler le parseur des dizaines de fois sans toucher au serveur — et
c'est la seule protection réelle contre le fait de marteler un site par inadvertance.

**Le délai ensuite**, 1,5 s plus un aléa, non désactivable en dessous d'une seconde.

**L'identification enfin** : un `User-Agent` qui dit qui fait ça et comment le joindre.

In [ ]:
import hashlib
import logging
import random
import time
import warnings
from pathlib import Path
from urllib.parse import quote, urlencode

warnings.filterwarnings("ignore", message=".*OpenSSL.*")
log = logging.getLogger("collecte")


def browse_url(category=None, page=1, series=None):
    """Construit une URL de liste conforme au robots.txt."""
    path = "/apps/modules"
    if category:
        path += "/category/" + quote(category, safe="")
    path += "/browse"
    if page > 1:
        path += f"/page/{page}"
    query = urlencode({"series": series}) if series else ""
    return f"{BASE}{path}" + (f"?{query}" if query else "")


class PoliteClient:
    """Telecharge des pages en respectant un delai et un cache disque."""

    def __init__(self, cache_dir, delay=1.5, jitter=0.5, timeout=30, max_retries=4):
        if delay < 1.0:
            raise ValueError("delay < 1 s : ce n'est pas poli, et c'est detectable.")
        self.cache_dir = Path(cache_dir)
        self.cache_dir.mkdir(parents=True, exist_ok=True)
        self.delay, self.jitter = delay, jitter
        self.timeout, self.max_retries = timeout, max_retries
        self._last = 0.0
        self.session = requests.Session()
        self.session.headers.update({"User-Agent": USER_AGENT,
                                     "Accept-Language": "en"})
        self.stats = {"hits": 0, "misses": 0, "errors": 0}

    def _cache_path(self, url):
        return self.cache_dir / (hashlib.sha256(url.encode()).hexdigest()[:20] + ".html")

    def _wait(self):
        pause = self.delay + random.uniform(0, self.jitter) - (time.monotonic() - self._last)
        if pause > 0:
            time.sleep(pause)
        self._last = time.monotonic()

    def get(self, url, use_cache=True):
        path = self._cache_path(url)
        if use_cache and path.exists():
            self.stats["hits"] += 1
            return path.read_text(encoding="utf-8")

        for essai in range(1, self.max_retries + 1):
            self._wait()
            try:
                r = self.session.get(url, timeout=self.timeout)
            except requests.RequestException as exc:
                log.warning("%s : %s (essai %d)", url, exc, essai)
            else:
                if r.status_code == 200:
                    self.stats["misses"] += 1
                    path.write_text(r.text, encoding="utf-8")
                    return r.text
                if r.status_code == 404:
                    self.stats["errors"] += 1
                    return None
                log.warning("%s : HTTP %d (essai %d)", url, r.status_code, essai)
            time.sleep(min(60, 2 ** essai))  # repli exponentiel

        self.stats["errors"] += 1
        return None


client = PoliteClient("../data/raw/cache_nb")
print("client pret")

## 3. Le piège : la liste des catégories

La page d'accueil affiche **18 catégories**. Les prendre pour la liste complète est
l'erreur naturelle — et surtout, elle est **silencieuse** : rien sur le site ne signale
qu'il en manque.

Le compteur « N Apps found » affiché sur chaque page de liste permet de la détecter.

In [ ]:
import html as html_mod

RESUME_RE = re.compile(r'<span class="[^"]*loempia_search_resume[^"]*">(.*?)</span>', re.S)
TAG_RE = re.compile(r"<[^>]+>")
COUNT_RE = re.compile(r"([\d\s,\u202f\u00a0]+?)\s*Apps?\s+found", re.I)


def parse_count(page_html):
    """Extrait l'entier du libelle << N Apps found >>.

    Le nombre et le libelle sont separes par des sauts de ligne et parfois une
    balise : chercher dans le HTML brut echoue des qu'un <b> s'intercale. On
    reduit donc au bloc du compteur, puis on retire les balises.
    """
    bloc = RESUME_RE.search(page_html)
    texte = TAG_RE.sub(" ", bloc.group(1) if bloc else page_html)
    m = COUNT_RE.search(texte)
    if not m:
        return None
    chiffres = re.sub(r"[^\d]", "", m.group(1))
    return int(chiffres) if chiffres else None


total_site = parse_count(client.get(browse_url()))
print(f"Total annonce par le site : {total_site:,} modules")

In [ ]:
# Les 18 categories affichees sur la page d'accueil.
CATEGORIES_VEDETTES = [
    "Accounting", "Discuss", "Document Management", "Extra Tools",
    "Human Resources", "Industries", "Localization", "Manufacturing",
    "Marketing", "Point of Sale", "Productivity", "Project", "Purchases",
    "Sales", "Tutorial", "Warehouse", "Website", "eCommerce",
]

effectifs_vedettes = {}
for nom in CATEGORIES_VEDETTES:
    effectifs_vedettes[nom] = parse_count(client.get(browse_url(category=nom))) or 0

somme = sum(effectifs_vedettes.values())
print(f"Somme des 18 categories  : {somme:,}")
print(f"Total annonce par le site: {total_site:,}")
print(f"Manquants                : {total_site - somme:,} "
      f"({(total_site - somme) / total_site * 100:.0f} %)")

### Un tiers du catalogue est ailleurs

Environ **32 %** des modules n'apparaissent dans aucune des 18 catégories affichées.
Ce ne sont pas des modules « sans catégorie » : ils sont rangés dans des catégories
que le site n'expose sur aucune page de navigation.

On les retrouve en sondant des noms candidats, tirés des catégories standard d'Odoo
(`ir.module.category`). Sonder large coûte une requête par nom et rate moins qu'une
liste prudente.

In [ ]:
CANDIDATES = sorted(set(CATEGORIES_VEDETTES + [
    "Tools", "Services", "Technical", "Uncategorized", "Administration", "Hidden",
    "Theme", "Themes", "Generic Modules", "Inventory", "CRM", "Helpdesk",
    "Field Service", "Maintenance", "Quality", "Fleet", "Events", "Timesheets",
    "Knowledge", "Sign", "Studio", "Internet of Things", "IoT", "Lunch",
    "Mass Mailing", "Notes", "Repair", "Subscription", "Subscriptions",
    "Website Theme", "Finance", "Payroll", "Recruitment", "Appraisals", "Time Off",
    "Expenses", "Invoicing", "Purchase", "Planning", "Appointment", "Surveys",
    "Social Marketing", "Email Marketing", "SMS Marketing", "Marketing Automation",
    "eLearning", "Forum", "Blog", "Live Chat", "Rental", "PLM", "Barcode",
    "Data Cleaning", "Spreadsheet", "Dashboards", "Reporting", "Localisation",
    "Connector", "Integration", "Payment", "Payment Providers", "Shipping",
    "Delivery", "Education", "Healthcare", "Real Estate", "Construction",
    "Restaurant", "Hotel", "Agriculture", "Transport", "Logistics", "Nonprofit",
    "Government", "Artificial Intelligence", "Security", "Extra Rights",
    "Usability", "Customizations", "Migration", "Backup", "API", "Mobile",
    "Other", "Others", "Miscellaneous", "Utilities", "Widgets", "Point Of Sale",
    "Ecommerce", "VoIP", "WhatsApp", "Documents", "Approvals", "Frontend",
    "Backend", "Portal",
]))

trouvees = {}
for i, nom in enumerate(CANDIDATES, 1):
    n = parse_count(client.get(browse_url(category=nom)) or "")
    if n:
        trouvees[nom] = n
    print(f"\r[{i:3d}/{len(CANDIDATES)}] {len(trouvees)} categories non vides", end="")

print(f"\n\n{len(trouvees)} categories non vides trouvees")

### Valider la liste

Une liste de catégories n'a de valeur que si l'on peut montrer qu'elle est complète.
Le test est simple : **la somme des effectifs doit rejoindre le total annoncé**.

Un point de détail compte ici : les URL de catégorie sont **insensibles à la casse**.
`Point Of Sale` et `Point of Sale` désignent la même page et seraient comptés deux
fois. Il faut dédoublonner avant de sommer.

In [ ]:
import pandas as pd

cats = (pd.DataFrame(sorted(trouvees.items()), columns=["category", "n_modules"])
          .assign(cle=lambda d: d.category.str.lower()))

doublons = cats[cats.duplicated("cle", keep=False)]
print("Doublons de casse :")
print(doublons[["category", "n_modules"]].to_string(index=False))

cats = (cats.drop_duplicates("cle", keep="first")
            .drop(columns="cle")
            .sort_values("n_modules", ascending=False)
            .reset_index(drop=True))

residu = total_site - cats.n_modules.sum()
print(f"\nCategories retenues : {len(cats)}")
print(f"Somme des effectifs : {cats.n_modules.sum():,}")
print(f"Total annonce       : {total_site:,}")
print(f"Residu non attribue : {residu:,}  ({residu / total_site * 100:.2f} %)")

cats.to_csv("../data/processed/categories.csv", index=False)

**Le résidu tombe à 0,08 %.**

La liste est donc complète, à 74 modules près sur 90 848.

**Attention à ne pas en déduire trop.** Il serait tentant de conclure que les
catégories forment une partition — puisque leur somme ne dépasse pas le total. Ce
serait une erreur, et la collecte le montrera : **23,4 % des modules relèvent de
plusieurs catégories**, jusqu'à treize pour l'un d'eux.

L'explication est que les deux nombres comparés ici comptent la même chose : des
**appartenances**, pas des modules. La liste non filtrée du store est elle aussi
paginée sur les couples (module, catégorie). Les deux totaux coïncident parce qu'ils
sont gonflés de la même façon.

C'est exactement le genre de raccourci qu'un contrôle de cohérence ne détecte pas :
les chiffres s'accordent, et la conclusion est fausse. Le dédoublonnage est fait au
notebook suivant.

In [ ]:
print(f"Les 15 plus grosses categories, sur {len(cats)} :\n")
for r in cats.head(15).itertuples():
    barre = "#" * round(r.n_modules / cats.n_modules.max() * 50)
    vedette = " " if r.category in CATEGORIES_VEDETTES else "*"
    print(f"{vedette} {r.category:<22}{r.n_modules:>7,}  {barre}")
print("\n* = categorie invisible depuis la page d'accueil")

`Tools`, **la plus grosse catégorie du site avec 12 210 modules**, n'est affichée nulle
part. C'est elle, à elle seule, qui explique une bonne part du tiers manquant.

## 4. Le recensement catégorie × version

L'axe temporel est le plus précieux du projet : si une fonction concentrait beaucoup de
modules en version 14 et peu en version 19, l'éditeur a absorbé le besoin ; si elle
reste dense, le manque persiste.

**Et c'est ici, pas dans les fiches, qu'il faut aller le chercher.** La liste non
filtrée n'affiche chaque module **qu'une fois, à sa version la plus récente** : 55 %
du corpus apparaît en 19.0, et seuls 260 modules sur 72 316 y figurent sous plus d'une
version. Cette colonne décrit la fraîcheur du catalogue, pas sa couverture par
version — s'en servir pour l'analyse temporelle donnerait un résultat faux.

Le filtre `?series=`, lui, recense les modules **disponibles** pour une version :
277 538 disponibilités module-version, contre 72 579 lignes dans le corpus.

On interroge donc le compteur pour les 111 catégories × 16 séries, soit 1 776
requêtes. C'est la seule source qui réponde correctement aux questions 7 et 8.

In [ ]:
SERIES = ["5.0", "6.0", "6.1", "7.0", "8.0", "9.0", "10.0", "11.0",
          "12.0", "13.0", "14.0", "15.0", "16.0", "17.0", "18.0", "19.0"]

lignes = []
total_appels = len(cats) * len(SERIES)
fait = 0

for cat in cats.category:
    for serie in SERIES:
        url = browse_url(category=cat, series=serie)
        n = parse_count(client.get(url) or "")
        lignes.append({"category": cat, "series": serie, "n_modules": n})
        fait += 1
        if fait % 50 == 0 or fait == total_appels:
            print(f"\r[{fait:4d}/{total_appels}] {cat:<24} {serie:<5}   ", end="")

census = pd.DataFrame(lignes)
census.to_csv("../data/interim/census_full.csv", index=False)

total_dispo = census.n_modules.sum()
print(f"\n\n{len(census):,} releves ecrits")
print(f"Disponibilites module-version recensees : {total_dispo:,.0f}")
print(f"(a comparer aux {total_site:,} modules de la liste non filtree :")
print(f" un module disponible pour six versions compte six fois ici)")

## 5. Les fiches modules

Chaque page de liste contient 20 cartes, et **la carte porte déjà l'essentiel** : nom
technique, série Odoo, titre, éditeur, prix, résumé, compteur d'usage. Parcourir les
listes plutôt que les 90 000 pages de détail divise le volume de requêtes par 20.

La contrepartie est assumée : la note moyenne, la licence et la date de publication ne
figurent pas sur les cartes. C'est une limite à documenter, pas un défaut de collecte.

Deux variantes de carte coexistent et ne doivent pas être confondues :

| | Prix | Compteur |
|---|---|---|
| Module **gratuit** | `FREE` | `Total Downloads: N` |
| Module **payant** | montant + devise | `Total Purchases: N` |

Téléchargements et achats **ne sont pas la même grandeur**. Les additionner produirait
un chiffre qui ne veut rien dire : on les garde dans deux colonnes distinctes.

In [ ]:
from dataclasses import asdict, dataclass
from typing import Optional

CARD_SPLIT_RE = re.compile(r'<div class="\s*loempia_app_entry loempia_app_card')
# Le segment de serie n'est pas toujours << 18.0 >> : l'Apps Store publie aussi des
# releases intermediaires d'Odoo Online sous la forme << saas-19.4 >>. Un motif
# limite a \d+\.\d+ les rejetait silencieusement (3 cartes sur 20 sur certaines pages).
HREF_RE = re.compile(r'href="/apps/modules/([\w.-]+)/([^"/?#]+)"')
SUMMARY_RE = re.compile(r'<p class="loempia_panel_summary">(.*?)</p>', re.S)
TITLE_RE = re.compile(r'<h5 title="(.*?)"', re.S)
AUTHOR_RE = re.compile(r'loempia_panel_author[^>]*>\s*<b>(.*?)</b>', re.S)
PRICE_BLOCK_RE = re.compile(r'loempia_panel_price(.*?)</div>', re.S)
CURRENCY_RE = re.compile(r'<span class="oe_currency_value">([\d.,]+)</span>\s*([^<\s]*)', re.S)
IMAGE_RE = re.compile(r'module_image\?image_id=(\d+)')
COUNTER_RE = re.compile(r'title="Total (Downloads|Purchases):\s*([\d,\s]+),\s*Last month:\s*([\d,\s]+)"', re.S)


def _texte(fragment):
    return re.sub(r"\s+", " ", html_mod.unescape(TAG_RE.sub(" ", fragment))).strip()


def _entier(brut):
    chiffres = re.sub(r"[^\d]", "", brut)
    return int(chiffres) if chiffres else None


@dataclass
class Card:
    technical_name: str
    series: str
    series_kind: str
    title: str
    author: str
    is_free: bool
    price_value: Optional[float]
    price_currency: Optional[str]
    n_downloads: Optional[int]
    n_purchases: Optional[int]
    n_last_month: Optional[int]
    summary: str
    image_id: Optional[str]
    source_category: str
    source_url: str


def parse_page(page_html, source_category, source_url):
    cartes = []
    for bloc in CARD_SPLIT_RE.split(page_html)[1:]:
        href = HREF_RE.search(bloc)
        if not href:
            continue  # une carte sans lien module n'a pas de cle exploitable
        serie, nom = href.group(1), href.group(2)

        resume = SUMMARY_RE.search(bloc)
        titre = TITLE_RE.search(bloc)
        auteur = AUTHOR_RE.search(bloc)

        gratuit, prix, devise = False, None, None
        bloc_prix = PRICE_BLOCK_RE.search(bloc)
        if bloc_prix:
            montant = CURRENCY_RE.search(bloc_prix.group(1))
            if montant:
                prix = float(montant.group(1).replace(",", ""))
                devise = html_mod.unescape(montant.group(2)).strip() or None
            elif "FREE" in _texte(bloc_prix.group(1)).upper():
                gratuit, prix = True, 0.0

        dl = achats = mois = None
        compteur = COUNTER_RE.search(bloc)
        if compteur:
            genre, total, dernier = compteur.groups()
            mois = _entier(dernier)
            if genre == "Downloads":
                dl = _entier(total)
            else:
                achats = _entier(total)

        img = IMAGE_RE.search(bloc)
        cartes.append(Card(
            technical_name=nom, series=serie,
            series_kind="saas" if serie.startswith("saas") else "major",
            title=html_mod.unescape(titre.group(1)).strip() if titre else "",
            author=_texte(auteur.group(1)) if auteur else "",
            is_free=gratuit, price_value=prix, price_currency=devise,
            n_downloads=dl, n_purchases=achats, n_last_month=mois,
            summary=_texte(resume.group(1)) if resume else "",
            image_id=img.group(1) if img else None,
            source_category=source_category, source_url=source_url))
    return cartes


# Controle : une page de liste doit rendre exactement 20 cartes.
essai = parse_page(client.get(browse_url(category="Accounting")), "Accounting", "test")
print(f"{len(essai)} cartes extraites (attendu : 20)")
print(f"\nExemple :")
for k, v in asdict(essai[0]).items():
    print(f"  {k:<16} {str(v)[:64]}")

### La collecte complète

~4 650 pages, environ deux heures à 1,5 s par requête. Chaque page est écrite
immédiatement et son URL journalisée : le notebook peut être interrompu et repris sans
rien reperdre, et le cache évite tout retéléchargement.

> **Cette cellule est longue.** Pour la tester d'abord, mettre `LIMITE_PAGES = 3`.

In [ ]:
import csv

LIMITE_PAGES = None   # None = tout ; un entier pour tester
SORTIE = Path("../data/raw/modules_raw.csv")
JOURNAL = Path("../data/raw/pages_done.txt")

SORTIE.parent.mkdir(parents=True, exist_ok=True)
deja = set(JOURNAL.read_text().split("\n")) - {""} if JOURNAL.exists() else set()
print(f"Reprise : {len(deja)} pages deja collectees")

champs = list(Card.__dataclass_fields__)
with SORTIE.open("a", newline="", encoding="utf-8") as fh, JOURNAL.open("a") as jf:
    writer = csv.DictWriter(fh, fieldnames=champs)
    if SORTIE.stat().st_size == 0:
        writer.writeheader()

    for i, cat in enumerate(cats.category, 1):
        total_cat = int(cats.loc[cats.category == cat, "n_modules"].iloc[0])
        n_pages = -(-total_cat // 20)
        if LIMITE_PAGES:
            n_pages = min(n_pages, LIMITE_PAGES)

        for page in range(1, n_pages + 1):
            url = browse_url(category=cat, page=page)
            if url in deja:
                continue
            page_html = client.get(url)
            if page_html is None:
                continue
            writer.writerows(asdict(c) for c in parse_page(page_html, cat, url))
            fh.flush()
            jf.write(url + "\n")
            jf.flush()
            print(f"\r[{i:3d}/{len(cats)}] {cat:<24} page {page:>4}/{n_pages}   ", end="")

print(f"\n\nTermine. Cache : {client.stats}")

In [ ]:
brut = pd.read_csv(SORTIE)
print(f"{len(brut):,} lignes collectees, {len(brut.columns)} colonnes")
print(f"Modules distincts : {brut.technical_name.nunique():,}")
print(f"Series            : {brut.series.nunique()}")
print(f"\nTaux de remplissage (%) :")
print((brut.notna().mean() * 100).round(1).to_string())

---

**Sorties de ce notebook**

| Fichier | Contenu |
|---|---|
| `data/processed/categories.csv` | les 111 catégories et leurs effectifs |
| `data/interim/census_category_series.csv` | la matrice catégorie × version |
| `data/raw/modules_raw.csv` | les fiches modules, brutes |

Suite : **`02_nettoyage.ipynb`**.